In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Load and preprocess MNIST dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Define CNN model
cnn_model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(20, (5, 5), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(20, (4, 4), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(20, (4, 4), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(200, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Train CNN
cnn_model.fit(x_train, y_train, epochs=5, batch_size=128, verbose=1, validation_split=0.1)

# Evaluate model on clean test data
accuracy_clean = cnn_model.evaluate(x_test, y_test, verbose=0)[1] * 100
print(f"Accuracy on clean test images: {accuracy_clean:.2f}%")

# FGSM Attack Function
def FGSM(model, image, label, eps=0.1):
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        pred = model(image)
        loss = tf.keras.losses.CategoricalCrossentropy()(label, pred)
    gradient = tape.gradient(loss, image)
    adversary = image + eps * tf.sign(gradient)
    return tf.clip_by_value(adversary, 0, 1)

# Generate FGSM adversarial images
num_samples = 1000
x_test_adv = np.array([FGSM(cnn_model, x_test[i:i+1], y_test[i:i+1]).numpy() for i in range(num_samples)])
x_test_adv = x_test_adv.reshape(-1, 28, 28, 1)

# Evaluate model on adversarial images
accuracy_adv = cnn_model.evaluate(x_test_adv, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after FGSM attack: {accuracy_adv:.2f}%")

# Apply Random Gaussian Noise (RGN) to adversarial images
def apply_gaussian_noise(images, sigma_min=0.0005, sigma_max=0.005):
    noise_std = np.random.uniform(sigma_min, sigma_max, images.shape)
    noisy_images = images + np.random.normal(0, noise_std)
    return np.clip(noisy_images, 0, 1)

x_test_adv_noisy = apply_gaussian_noise(x_test_adv)

# Evaluate model on noise-injected adversarial images
accuracy_noisy = cnn_model.evaluate(x_test_adv_noisy, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after applying Random Gaussian Noise: {accuracy_noisy:.2f}%")

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Load and preprocess MNIST dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Define CNN model
cnn_model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(20, (5, 5), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(20, (4, 4), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv2D(20, (4, 4), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(200, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [2]:
# Train CNN
cnn_model.fit(x_train, y_train, epochs=10, batch_size=128, verbose=1, validation_split=0.1)


Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 24s 29ms/step - accuracy: 0.8858 - loss: 0.4880 - val_accuracy: 0.9663 - val_loss: 0.1163
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9841 - loss: 0.0536 - val_accuracy: 0.9892 - val_loss: 0.0458
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9882 - loss: 0.0366 - val_accuracy: 0.9837 - val_loss: 0.0885
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9905 - loss: 0.0304 - val_accuracy: 0.9892 - val_loss: 0.0472
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9931 - loss: 0.0216 - val_accuracy: 0.9853 - val_loss: 0.0731
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9938 - loss: 0.0189 - val_accuracy: 0.9875 - val_loss: 0.0543
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9940 - loss: 0.0187 - val_accuracy: 0.9902 - val_loss: 0.0422
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9948 - loss: 0.0148 - val_accuracy:

In [3]:

# Evaluate model on clean test data
accuracy_clean = cnn_model.evaluate(x_test, y_test, verbose=0)[1] * 100
print(f"Accuracy on clean test images: {accuracy_clean:.2f}%")

Accuracy on clean test images: 98.71%


In [4]:
# FGSM Attack Function
def FGSM(model, image, label, eps=0.1):
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        pred = model(image)
        loss = tf.keras.losses.CategoricalCrossentropy()(label, pred)
    gradient = tape.gradient(loss, image)
    adversary = image + eps * tf.sign(gradient)
    return tf.clip_by_value(adversary, 0, 1)

# Generate FGSM adversarial images
num_samples = 1000
x_test_adv = np.array([FGSM(cnn_model, x_test[i:i+1], y_test[i:i+1]).numpy() for i in range(num_samples)])
x_test_adv = x_test_adv.reshape(-1, 28, 28, 1)

# Evaluate model on adversarial images
accuracy_adv = cnn_model.evaluate(x_test_adv, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after FGSM attack: {accuracy_adv:.2f}%")

Accuracy after FGSM attack: 58.10%


In [5]:
# Apply Random Gaussian Noise (RGN) to adversarial images
def apply_gaussian_noise(images, sigma_min=0.0005, sigma_max=0.005):
    noise_std = np.random.uniform(sigma_min, sigma_max, images.shape)
    noisy_images = images + np.random.normal(0, noise_std)
    return np.clip(noisy_images, 0, 1)

x_test_adv_noisy = apply_gaussian_noise(x_test_adv)

# Evaluate model on noise-injected adversarial images
accuracy_noisy = cnn_model.evaluate(x_test_adv_noisy, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after applying Random Gaussian Noise: {accuracy_noisy:.2f}%")

Accuracy after applying Random Gaussian Noise: 58.50%


In [6]:
# Apply Stochastic Motion Blur (SMB)
def apply_motion_blur(images, kernel_size=3):
    import cv2
    blurred_images = np.zeros_like(images)
    for i in range(images.shape[0]):
        kernel = np.zeros((kernel_size, kernel_size))
        kernel[:, kernel_size // 2] = 1 / kernel_size
        blurred_images[i, :, :, 0] = cv2.filter2D(images[i, :, :, 0], -1, kernel)
    return np.clip(blurred_images, 0, 1)

x_test_adv_smb = apply_motion_blur(x_test_adv_noisy)
accuracy_smb = cnn_model.evaluate(x_test_adv_smb, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after applying Stochastic Motion Blur: {accuracy_smb:.2f}%")

Accuracy after applying Stochastic Motion Blur: 67.40%


In [7]:
# Apply Stochastic Glass Blur (SGB)
def apply_glass_blur(images, sigma=0.7):
    from scipy.ndimage import gaussian_filter
    blurred_images = gaussian_filter(images, sigma=(0, sigma, sigma, 0))
    return np.clip(blurred_images, 0, 1)

x_test_adv_sgb = apply_glass_blur(x_test_adv_smb)
accuracy_sgb = cnn_model.evaluate(x_test_adv_sgb, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after applying Stochastic Glass Blur: {accuracy_sgb:.2f}%")


Accuracy after applying Stochastic Glass Blur: 67.10%


In [8]:
# Apply Random Sized Coarse Dropout (RSCD)
def apply_coarse_dropout(images, drop_prob=0.1):
    mask = np.random.rand(*images.shape) > drop_prob
    return images * mask

x_test_adv_rscd = apply_coarse_dropout(x_test_adv_sgb)
accuracy_rscd = cnn_model.evaluate(x_test_adv_rscd, y_test[:num_samples], verbose=0)[1] * 100
print(f"Accuracy after applying Random Sized Coarse Dropout: {accuracy_rscd:.2f}%")


Accuracy after applying Random Sized Coarse Dropout: 61.60%
